# Self-Healing Feeder: Active-Grid GNN + Conformal Safety on Kaggle GPU

Runs `research/active-grid-gnn/PLAN.md` end-to-end on Kaggle (offline OpenDSS wheels + PyTorch CUDA GPU):
1. Active OpenDSS fault & non-fault transient sweeps on IEEE 123-bus (4.16 kV) and IEEE 34-bus (24.9 kV)
   with 0-50% current-limited inverter DER (`Model=7`), dynamic reconfiguration + closed-loop topologies,
   voltage-normalized HIFs (up to 10 kOhm on 24.9 kV), and non-fault distractors (cap switch, PV cloud, load step).
2. Trains FPI, `ActivePerNodeGBDT`, and 3-layer bidirectional `ActiveGridGATv2` strictly on IEEE 123-bus (5-fold GroupKFold by bus).
3. Evaluates zero-shot transfer, in-domain upper bound, and 5%/20% few-shot on unseen active IEEE 34-bus.
4. Runs Split Conformal Prediction (APS at 90% and 95% target coverage) and records `provenance`.
5. Post-hoc (PLAN.md Addendum A): compares deterministic APS, LAC and randomized APS on the same probabilities.

Simulation only. Nothing here is field data.

In [ ]:
import os, shutil, zipfile

DATASET_DIR = "/kaggle/input"
WORK = "/kaggle/working/self-healing-feeder"
if os.path.exists(WORK):
    shutil.rmtree(WORK)

src = next((root for root, _dirs, files in os.walk(DATASET_DIR) if "run.py" in files), None)
if src is None:
    zips = [os.path.join(r, f) for r, _d, fs in os.walk(DATASET_DIR) for f in fs if f.endswith(".zip")]
    for z in zips:
        zipfile.ZipFile(z).extractall("/kaggle/working/_bundle")
    src = next((root for root, _dirs, files in os.walk("/kaggle/working/_bundle") if "run.py" in files), None)
if src is None:
    raise SystemExit("Could not find project bundle in /kaggle/input")
shutil.copytree(src, WORK)
shutil.rmtree("/kaggle/working/_bundle", ignore_errors=True)
os.chdir(WORK)
print("Working dir:", WORK)
print("Top-level:", sorted(os.listdir(WORK)))

In [ ]:
# Install OpenDSS offline from bundled Linux wheels (no Internet required).
import subprocess, sys
if os.path.isdir(os.path.join(WORK, "wheels")):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index",
                    "--find-links", "wheels", "OpenDSSDirect.py==0.9.4"], check=True)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "OpenDSSDirect.py==0.9.4"], check=True)

import torch, opendssdirect as dss
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("OpenDSS:", dss.Basic.Version())

In [ ]:
# Fast verification tests on Kaggle environment before running the full experiment.
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_active_grid.py"], check=True)

In [ ]:
# Run the full Active-Grid GNN + Split Conformal Prediction study.
subprocess.run([sys.executable, "-u", "scripts/active_grid_run.py"], check=True)

In [ ]:
# Post-hoc conformal comparison (PLAN.md Addendum A): same probabilities and splits as above.
subprocess.run([sys.executable, "-u", "scripts/active_conformal_posthoc.py"], check=True)

In [ ]:
# Package results/active_grid, both run records and the conformal input probabilities into /kaggle/working.
import json
from pathlib import Path

OUT = Path("/kaggle/working/active_grid_results")
if OUT.exists():
    shutil.rmtree(OUT)
(OUT / "active_grid").mkdir(parents=True)
(OUT / "provenance").mkdir(parents=True)

for p in sorted((Path(WORK) / "results" / "active_grid").glob("*.json")):
    shutil.copy2(p, OUT / "active_grid" / p.name)
for prod in ("active_grid", "active_conformal_posthoc"):
    shutil.copy2(Path(WORK) / "results" / "provenance" / f"{prod}.json", OUT / "provenance" / f"{prod}.json")
(OUT / "generated").mkdir(parents=True)
shutil.copy2(Path(WORK) / "data" / "generated" / "active_probs.npz", OUT / "generated" / "active_probs.npz")

os.chdir("/kaggle/working")
shutil.rmtree(WORK)
shutil.make_archive("/kaggle/working/active_grid_results", "zip", OUT)

p34 = json.loads((OUT / "active_grid" / "phase_active_ieee34.json").read_text())
print("=== Pre-committed criteria summary ===")
print(json.dumps(p34["pre_committed_criteria"], indent=2))
print("Saved /kaggle/working/active_grid_results.zip")